# Vibe Quest S9 · Manifold learning y PCA

**Pregunta elegida:** Explora el concepto de "manifold learning" en autoencoders y cómo se relaciona con PCA. ¿Cuáles son las implicaciones teóricas y prácticas de estas diferencias?

Autor: Jorge Chamorro · INF659 Técnicas Avanzadas de Data Mining y Sistemas Inteligentes · PUCP 2026-2

Uso de IA: investigué con un asistente de IA y revisé las fuentes citadas al final. La responsabilidad de lo presentado es mía.

In [ ]:
# ===== 1. Librerías y configuración =====
# Todo corre en CPU (PyTorch, redes pequeñas) y en pocos minutos. Antes de la presentación se ejecuta el cuaderno completo (Entorno de ejecución > Ejecutar todo).
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from sklearn.datasets import make_swiss_roll
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
SEMILLA = 0                                # misma semilla = mismos resultados en cada ejecución
torch.manual_seed(SEMILLA)                 # fija también la inicialización de los pesos de las redes
torch.set_num_threads(2)                   # redes pequeñas: pocos hilos bastan en CPU
NIVELES_RUIDO = [0.0, 0.15, 0.3, 0.6]      # ruido que se agrega a los datos (en desviaciones estándar)
DIMENSIONES = [1, 2, 3]                    # dimensión del espacio latente (k) que se puede elegir
plt.rcParams["figure.dpi"] = 90

In [ ]:
# ===== 2. Datos: el "rollo suizo" =====
# Es una hoja de papel (2 dimensiones) enrollada dentro del espacio 3D.
# Es un ejemplo clásico de variedad curva (Tenenbaum et al., 2000). Los datos son sintéticos: no vienen de las fuentes, que usan imágenes (Fournier y Aloise) o física (Wetzel).
# Cada punto tiene un valor t que indica su posición a lo largo de la hoja; solo se usa para colorear.
X_limpio, t = make_swiss_roll(n_samples=1500, noise=0.0, random_state=SEMILLA)

# Se estandariza para que los errores de todos los métodos se midan en la misma escala.
X_limpio = StandardScaler().fit_transform(X_limpio)

# Ruido fijo: se genera una sola vez para que cada nivel de ruido use el mismo patrón y solo cambie su intensidad.
ruido_base = np.random.default_rng(SEMILLA).normal(size=X_limpio.shape)

def datos(ruido):
    # Devuelve train y test (80/20) para un nivel de ruido.
    # El error siempre se mide en test: son puntos que el método no vio al ajustarse.
    X = X_limpio + ruido * ruido_base
    return train_test_split(X, t, test_size=0.2, random_state=SEMILLA)

def dibujar_3d(ax, X, color, titulo, s=5):
    # Dibujo 3D de una nube de puntos coloreada por t.
    ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=color, s=s, cmap="Spectral")
    ax.set_title(titulo, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.view_init(elev=12, azim=-70)

Xtr, Xte, ttr, tte = datos(0.0)
fig = plt.figure(figsize=(4.5, 4))
dibujar_3d(fig.add_subplot(projection="3d"), Xte, tte, "Datos originales (rollo suizo)")
plt.show()

In [ ]:
# ===== 3. Los tres métodos que se comparan =====
# (a) PCA: la mejor proyección LINEAL a k dimensiones (un plano si k=2).
# (b) Autoencoder lineal (PyTorch): una red que comprime a k números y reconstruye, sin activaciones. Base teórica: Baldi y Hornik (1989).
# (c) Autoencoder no lineal: igual, pero con capas tanh que le permiten doblarse y seguir superficies curvas.

class Autoencoder:
    # Autoencoder en PyTorch: codificador (3 -> k) + decodificador (k -> 3), entrenado para reproducir su entrada.
    # La capa de k neuronas es el "cuello de botella" = espacio latente.
    def __init__(self, k, tipo, epocas=2500, lr=0.005, semilla=SEMILLA):
        self.k, self.tipo, self.epocas, self.lr = k, tipo, epocas, lr
        torch.manual_seed(semilla)                     # misma semilla = misma inicialización
        if tipo == "lineal":
            # Sin activaciones: solo una proyección lineal a k dimensiones y su inversa aproximada.
            self.enc = nn.Sequential(nn.Linear(3, k))
            self.dec = nn.Sequential(nn.Linear(k, 3))
        else:
            # Con tanh: puede doblarse y seguir superficies curvas.
            self.enc = nn.Sequential(nn.Linear(3, 32), nn.Tanh(), nn.Linear(32, k), nn.Tanh())
            self.dec = nn.Sequential(nn.Linear(k, 32), nn.Tanh(), nn.Linear(32, 3))
        self.red = nn.Sequential(self.enc, self.dec)

    def fit(self, X):
        X = torch.tensor(X, dtype=torch.float32)
        optimizador = torch.optim.Adam(self.red.parameters(), lr=self.lr)          # Adam, visto en S4
        decaimiento = torch.optim.lr_scheduler.ExponentialLR(optimizador, gamma=0.9995)  # baja la tasa de aprendizaje de a poco
        perdida = nn.MSELoss()                                                      # error cuadrático medio
        self.red.train()
        for _ in range(self.epocas):                   # lote completo: son solo 1 200 puntos de entrenamiento
            optimizador.zero_grad()
            l = perdida(self.red(X), X)                # entrada = salida: así aprende a comprimir y reconstruir
            l.backward()                               # retropropagación (S3)
            optimizador.step()
            decaimiento.step()
        self.red.eval()
        return self

    def encode(self, X):
        with torch.no_grad():
            return self.enc(torch.tensor(X, dtype=torch.float32)).numpy().astype(np.float64)

    def decode(self, Z):
        with torch.no_grad():
            return self.dec(torch.tensor(Z, dtype=torch.float32)).numpy().astype(np.float64)

class ModeloPCA:
    # Envoltorio para que PCA tenga la misma interfaz (encode / decode) que el autoencoder.
    def __init__(self, k):
        self.k = k
    def fit(self, X):
        self.pca = PCA(n_components=self.k).fit(X)
        return self
    def encode(self, X):
        return self.pca.transform(X)
    def decode(self, Z):
        return self.pca.inverse_transform(Z)

def reconstruir(modelo, X):
    # Comprime a k dimensiones y vuelve a 3D.
    return modelo.decode(modelo.encode(X))

def error(X, Xrec):
    # Error cuadrático medio de reconstrucción: menor = conserva mejor la forma.
    return float(np.mean((X - Xrec) ** 2))

In [ ]:
# ===== 4. Entrenamiento previo (se hace una sola vez) =====
# Se entrena todo antes para que los controles interactivos respondan al instante.
# Son 4 niveles de ruido x 3 dimensiones x 3 métodos.
import time
modelos = {}          # modelos[(ruido, k, metodo)] = modelo ya ajustado
inicio = time.time()
for ruido in NIVELES_RUIDO:
    Xtr_, Xte_, _, _ = datos(ruido)
    for k in DIMENSIONES:
        modelos[(ruido, k, "PCA")] = ModeloPCA(k).fit(Xtr_)
        modelos[(ruido, k, "AE lineal")] = Autoencoder(k, "lineal").fit(Xtr_)
        modelos[(ruido, k, "AE no lineal")] = Autoencoder(k, "no lineal").fit(Xtr_)
    print(f"ruido {ruido}: listo ({time.time() - inicio:.0f} s acumulados)")

In [ ]:
# ===== 5. Resultado principal: error de reconstrucción según k (sin ruido) =====
# Se espera que: (1) el AE lineal dé casi el mismo error que PCA y (2) el AE no lineal dé un error menor cuando k es 1 o 2.
METODOS = ["PCA", "AE lineal", "AE no lineal"]
Xtr0, Xte0, ttr0, tte0 = datos(0.0)
tabla = {m: [error(Xte0, reconstruir(modelos[(0.0, k, m)], Xte0)) for k in DIMENSIONES] for m in METODOS}

print("Error de reconstrucción en test (ruido = 0)")
print(f"{'k':>3} " + " ".join(f"{m:>13}" for m in METODOS))
for i, k in enumerate(DIMENSIONES):
    print(f"{k:>3} " + " ".join(f"{tabla[m][i]:>13.4f}" for m in METODOS))

# Verificación automática de la equivalencia PCA = autoencoder lineal (tolerancia de 10 %).
# Con k=3 el error es casi 0 en todos los métodos, porque no se reduce nada (los datos ya son 3D).
for i, k in enumerate(DIMENSIONES[:2]):
    assert abs(tabla["AE lineal"][i] - tabla["PCA"][i]) / tabla["PCA"][i] < 0.10, "El AE lineal no coincide con PCA"
print("\nVerificado: el autoencoder lineal reproduce el error de PCA.")

fig, ax = plt.subplots(figsize=(5.5, 3.6))
for m, estilo in zip(METODOS, ["o-", "s--", "^-"]):
    ax.plot(DIMENSIONES, tabla[m], estilo, label=m)
ax.set_xticks(DIMENSIONES); ax.set_xlabel("dimensión latente k"); ax.set_ylabel("error de reconstrucción (test)")
ax.set_title("PCA = AE lineal; el AE no lineal reconstruye mejor"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ===== 5b. Costo computacional: tiempo de ajuste (k = 2, ruido = 0) =====
# Fournier y Aloise (2019) recomiendan PCA para prototipar porque entrena mucho más rápido que un autoencoder.
# Aquí se mide con los mismos datos: se vuelve a ajustar cada método y se cronometra (PCA se promedia sobre 20 ajustes).
Xtr_t, _, _, _ = datos(0.0)
tiempos = {}
repeticiones = {"PCA": 20}   # PCA tarda milisegundos: se promedia 20 ajustes para que la medición sea estable
for nombre, construir in [("PCA", lambda: ModeloPCA(2)),
                          ("AE lineal", lambda: Autoencoder(2, "lineal")),
                          ("AE no lineal", lambda: Autoencoder(2, "no lineal"))]:
    n = repeticiones.get(nombre, 1)
    t0 = time.perf_counter()
    for _ in range(n):
        construir().fit(Xtr_t)
    tiempos[nombre] = (time.perf_counter() - t0) / n

print(f"{'método':>13} {'segundos':>10} {'veces PCA':>11}")
for nombre, seg in tiempos.items():
    print(f"{nombre:>13} {seg:>10.3f} {seg / tiempos['PCA']:>11,.0f}")
# Lectura: PCA se resuelve de forma directa (descomposición en valores singulares); los autoencoders necesitan iterar miles de veces.
# Esa diferencia de costo es la parte práctica del compromiso: el autoencoder no lineal reconstruye mejor, pero cuesta más.
# Los segundos dependen de la máquina y varían entre ejecuciones; lo que se conserva es el orden de magnitud (milisegundos frente a segundos).


In [ ]:
# ===== 5c. Comparación con Isomap (Tenenbaum et al., 2000) =====
# Isomap es un método clásico de manifold learning: mide distancias a lo largo de la superficie y "desenrolla" el rollo.
# En sklearn no tiene decodificador, así que no se puede medir error de reconstrucción. En su lugar se miden dos cosas con k = 2:
#   - Spearman: qué tan bien una coordenada latente ordena los puntos según t (su posición en la hoja). 1 = la hoja quedó desenrollada.
#   - Confiabilidad (trustworthiness): qué parte de los vecinos cercanos en el latente también lo eran en 3D. 1 = perfecto.
# Se toma la mejor coordenada en valor absoluto porque el latente no es único (eje y signo arbitrarios).
from sklearn.manifold import Isomap, trustworthiness
from scipy.stats import spearmanr

def calidad(Z, Xte_, tte_):
    rho = max(abs(spearmanr(Z[:, j], tte_)[0]) for j in range(Z.shape[1]))
    return rho, trustworthiness(Xte_, Z, n_neighbors=10)

print(f"{'ruido':>6} {'método':>13} {'Spearman con t':>15} {'confiabilidad':>14} {'segundos':>9}")
for ruido in [0.0, 0.3]:
    Xtr_i, Xte_i, ttr_i, tte_i = datos(ruido)
    latentes = {m: (modelos[(ruido, 2, m)].encode(Xte_i), None) for m in METODOS}
    t0 = time.perf_counter()
    iso = Isomap(n_neighbors=10, n_components=2).fit(Xtr_i)       # 10 vecinos por punto: valor habitual para este tamaño
    seg_iso = time.perf_counter() - t0
    latentes["Isomap"] = (iso.transform(Xte_i), seg_iso)
    for nombre, (Z, seg) in latentes.items():
        rho, conf = calidad(Z, Xte_i, tte_i)
        texto_seg = f"{seg:>9.3f}" if seg is not None else f"{'(previo)':>9}"
        print(f"{ruido:>6} {nombre:>13} {rho:>15.2f} {conf:>14.3f} {texto_seg}")
# Lectura esperada: sin ruido Isomap desenrolla el rollo (Spearman alto); PCA y los autoencoders no. Con ruido suele degradarse,
# porque unos pocos puntos desviados crean atajos entre vueltas del rollo y las distancias sobre la superficie dejan de ser fiables.
# Bengio et al. (2013) señalan además que los métodos como Isomap no aprenden una función que se aplique de forma directa a puntos nuevos,
# mientras que un autoencoder sí (encoder/decoder), y por eso puede reconstruir.


In [ ]:
# ===== 6. Control interactivo A: comparar la reconstrucción =====
# Mueve k y el ruido; el botón elige qué autoencoder se compara contra PCA.
# Cada gráfico 3D muestra los puntos de test: originales, reconstruidos por PCA y reconstruidos por el autoencoder.
def comparar(k, ruido, tipo_ae):
    _, Xte, _, tte = datos(ruido)
    rec_pca = reconstruir(modelos[(ruido, k, "PCA")], Xte)
    rec_ae = reconstruir(modelos[(ruido, k, tipo_ae)], Xte)
    e_pca, e_ae = error(Xte, rec_pca), error(Xte, rec_ae)

    fig = plt.figure(figsize=(14, 3.8))
    dibujar_3d(fig.add_subplot(1, 4, 1, projection="3d"), Xte, tte, "Original")
    dibujar_3d(fig.add_subplot(1, 4, 2, projection="3d"), rec_pca, tte, f"PCA (k={k})")
    dibujar_3d(fig.add_subplot(1, 4, 3, projection="3d"), rec_ae, tte, f"{tipo_ae} (k={k})")
    ax = fig.add_subplot(1, 4, 4)
    ax.bar(["PCA", tipo_ae], [e_pca, e_ae], color=["#888888", "#d95f02"])
    ax.set_ylabel("error de reconstrucción"); ax.set_title("Error (menor es mejor)")
    ax.set_ylim(0, max(e_pca, e_ae, 0.05) * 1.25)
    for i, v in enumerate([e_pca, e_ae]):
        ax.text(i, v, f"{v:.3f}", ha="center", va="bottom")
    plt.tight_layout(); plt.show(); plt.close(fig)

w_k = widgets.IntSlider(value=2, min=1, max=3, step=1, description="k latente", continuous_update=False)
w_ruido = widgets.SelectionSlider(options=NIVELES_RUIDO, value=0.0, description="ruido", continuous_update=False)
w_tipo = widgets.ToggleButtons(options=["AE lineal", "AE no lineal"], value="AE no lineal", description="autoencoder")
salida_a = widgets.interactive_output(comparar, {"k": w_k, "ruido": w_ruido, "tipo_ae": w_tipo})
display(widgets.VBox([widgets.HBox([w_k, w_ruido]), w_tipo, salida_a]))

In [ ]:
# ===== 7. Control interactivo B: recorrer el espacio latente (k = 2) =====
# Los dos deslizadores mueven un punto dentro del espacio latente de 2 dimensiones.
# A la izquierda se ve a qué punto 3D corresponde (punto rojo). Los puntos celestes son la superficie
# que el método "dibuja" al decodificar toda una cuadrícula del latente:
# PCA produce un plano; el autoencoder no lineal produce una hoja curva que sigue el rollo.
def explorar(modelo_nombre, ruido, s1, s2):
    Xtr, _, ttr, _ = datos(ruido)
    modelo = modelos[(ruido, 2, modelo_nombre)]
    Ztr = modelo.encode(Xtr)
    lo, hi = Ztr.min(axis=0), Ztr.max(axis=0)          # rango del latente en los datos de entrenamiento

    # Cuadrícula de 20x20 sobre ese rango, decodificada de vuelta a 3D
    g1, g2 = np.meshgrid(np.linspace(lo[0], hi[0], 20), np.linspace(lo[1], hi[1], 20))
    hoja = modelo.decode(np.c_[g1.ravel(), g2.ravel()])

    # Punto elegido con los deslizadores (0 = mínimo del rango, 1 = máximo)
    z = np.array([[lo[0] + s1 * (hi[0] - lo[0]), lo[1] + s2 * (hi[1] - lo[1])]])
    punto = modelo.decode(z)

    fig = plt.figure(figsize=(11, 4.4))
    ax3 = fig.add_subplot(1, 2, 1, projection="3d")
    ax3.scatter(Xtr[:, 0], Xtr[:, 1], Xtr[:, 2], c="lightgray", s=3)
    ax3.scatter(hoja[:, 0], hoja[:, 1], hoja[:, 2], c="#4aa3df", s=6, alpha=0.6)
    ax3.scatter(punto[:, 0], punto[:, 1], punto[:, 2], c="red", s=120, edgecolors="k")
    ax3.set_title(f"{modelo_nombre}: superficie decodificada", fontsize=10)
    ax3.set_xticks([]); ax3.set_yticks([]); ax3.set_zticks([]); ax3.view_init(elev=12, azim=-70)

    ax2 = fig.add_subplot(1, 2, 2)
    ax2.scatter(Ztr[:, 0], Ztr[:, 1], c=ttr, cmap="Spectral", s=5)
    ax2.scatter(z[:, 0], z[:, 1], c="red", s=120, edgecolors="k")
    ax2.set_xlabel("latente 1"); ax2.set_ylabel("latente 2"); ax2.set_title("Espacio latente (color = posición t en la hoja)", fontsize=10)
    plt.tight_layout(); plt.show(); plt.close(fig)

w_modelo = widgets.ToggleButtons(options=["PCA", "AE lineal", "AE no lineal"], value="AE no lineal", description="método")
w_ruido_b = widgets.SelectionSlider(options=NIVELES_RUIDO, value=0.0, description="ruido", continuous_update=False)
w_s1 = widgets.FloatSlider(value=0.5, min=0, max=1, step=0.02, description="latente 1", continuous_update=False)
w_s2 = widgets.FloatSlider(value=0.5, min=0, max=1, step=0.02, description="latente 2", continuous_update=False)
salida_b = widgets.interactive_output(explorar, {"modelo_nombre": w_modelo, "ruido": w_ruido_b, "s1": w_s1, "s2": w_s2})
display(widgets.VBox([w_modelo, widgets.HBox([w_ruido_b, w_s1, w_s2]), salida_b]))

In [ ]:
# ===== 8. Resumen numérico y referencias =====
# Resumen: cuánto mejora el autoencoder no lineal frente a PCA en cada nivel de ruido (k = 2, error en test).
print(f"{'ruido':>6} {'PCA':>8} {'AE lineal':>10} {'AE no lineal':>13} {'mejora vs PCA':>14}")
for ruido in NIVELES_RUIDO:
    _, Xte_, _, _ = datos(ruido)
    e = {m: error(Xte_, reconstruir(modelos[(ruido, 2, m)], Xte_)) for m in METODOS}
    mejora = 100 * (1 - e["AE no lineal"] / e["PCA"])
    print(f"{ruido:>6} {e['PCA']:>8.3f} {e['AE lineal']:>10.3f} {e['AE no lineal']:>13.3f} {mejora:>13.0f}%")

# Limitaciones a tener presentes:
# - Los datos son sintéticos y de solo 3 dimensiones; con imágenes reales el comportamiento puede cambiar.
# - El latente del autoencoder no es único: otra semilla da otros ejes (por eso no se compara coordenada contra coordenada).
# - Ganar en reconstrucción no garantiza que el latente sea más útil: en clasificación con imágenes, Fournier y Aloise (2019) hallaron que PCA igualó a los autoencoders
#   cuando la dimensión era suficiente y fue unas 100 veces más rápido. Aquí se midió el tiempo (celda 5b), pero no la clasificación.
# - El latente del autoencoder no despliega el rollo (celda 5c: Isomap sí lo hace sin ruido); lo que se mide es mejor reconstrucción.
# - Los resultados del notebook usan una sola semilla; el experimento con 10 semillas está en codigo/experimento_semillas_pytorch.py.
#
# Referencias:
# Baldi, P. y Hornik, K. (1989). Neural networks and principal component analysis: Learning from examples without local minima. Neural Networks, 2(1), 53-58.
# Hinton, G. E. y Salakhutdinov, R. R. (2006). Reducing the dimensionality of data with neural networks. Science, 313(5786), 504-507.
# Jolliffe, I. T. y Cadima, J. (2016). Principal component analysis: a review and recent developments. Phil. Trans. R. Soc. A, 374(2065), 20150202.
# Fournier, Q. y Aloise, D. (2019). Empirical comparison between autoencoders and traditional dimensionality reduction methods. IEEE AIKE 2019, 211-214.
# Bengio, Y., Courville, A. y Vincent, P. (2013). Representation learning: A review and new perspectives. IEEE TPAMI, 35(8), 1798-1828.
# Tenenbaum, J. B., de Silva, V. y Langford, J. C. (2000). A global geometric framework for nonlinear dimensionality reduction. Science, 290(5500), 2319-2323. (Isomap; el rollo suizo es su ejemplo clásico de variedad curva.)